# 3D Taylor-Expanded Wakefields

The `TaylorWakefield` class models the full 3D wakefield of a structure through a
second-order Taylor expansion of the longitudinal point-charge wake function near
the reference axis, following
[Zagorodnov, Bane, and Stupakov (2015)](https://doi.org/10.1103/PhysRevSTAB.18.104401).
This is the same model implemented in [ocelot](https://github.com/ocelot-collab/ocelot)'s
`Wake` physics process, and `TaylorWakefield` reads and writes the same wake table
file format. The implementation here was benchmarked against ocelot: per-particle
kicks agree to machine precision for file-based tables.

The longitudinal wake for a source particle at transverse position $(x_s, y_s)$
and a witness particle at $(x_w, y_w)$, separated longitudinally by $s \ge 0$, is

$$w(x_s, y_s, x_w, y_w, s) = \sum_{a \le b} h_{ab}(s)\, u_a u_b,
\qquad u = (1,\, x_s,\, y_s,\, x_w,\, y_w)$$

so the full 3D wake is represented by a set of one-dimensional components
$h_{ab}(s)$. Transverse wakes follow from the Panofsky–Wenzel theorem. This gives
longitudinal **and** transverse (dipole, quadrupole) kicks — unlike the purely
longitudinal 1D wakefields elsewhere in this package.

Note that the structure length is baked into the wake table: wake amplitudes are
in V/C for the whole structure, and kicks are returned in eV/c (not eV/m).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from beamphysics.testing import pg_from_random_normal
from beamphysics.wakefields import TaylorWakefield

## Analytic corrugated parallel-plate (dechirper) wake

`TaylorWakefield.parallel_plate` implements the analytic surface-impedance model of
[Bane, Stupakov, and Zagorodnov (2016)](https://doi.org/10.1103/PhysRevAccelBeams.19.084401)
for a flat corrugated dechirper, with the beam possibly offset from the center
(a port of ocelot's `WakeTableParallelPlate`).

Here: a 2 m long structure with 1 mm full gap, with the beam 250 µm from the
upper plate (i.e. offset 250 µm from the center).

In [ ]:
wake = TaylorWakefield.parallel_plate(
    plate_distance=250e-6,  # distance from the beam to the nearest plate
    half_gap=500e-6,  # half distance between the plates
    corrugation_gap=250e-6,
    corrugation_period=500e-6,
    length=2.0,  # structure length
    sigma=10e-6,  # sets the tabulated s range (50 sigma)
    orientation="horizontal",
)
wake

Each component $h_{ab}(s)$ is available by its index pair. The index meaning is
0: constant, 1: $x_s$, 2: $y_s$, 3: $x_w$, 4: $y_w$. For example `(0, 0)` is the
monopole longitudinal wake and `(0, 4)` the vertical dipole wake:

In [ ]:
wake.plot()
plt.yscale("symlog")

## Wake potentials for a current profile

`wake_potential` convolves a single component with a current profile. The profile
uses the beamphysics convention: larger $z$ is the bunch head. The longitudinal
potential (key `(0, 0)`) is in V (negative = energy loss); dipole witness
components (`(0, 3)`, `(0, 4)`) apply the Panofsky–Wenzel integral and give the
transverse kick per unit witness offset in V/m.

In [ ]:
z = np.linspace(-100e-6, 100e-6, 1000)
sigma_z = 10e-6
peak_current = 1000  # A
current = peak_current * np.exp(-0.5 * (z / sigma_z) ** 2)
profile = np.column_stack([z, current])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key, label in [
    (axes[0], (0, 0), "Longitudinal wake potential (MV)"),
    (axes[1], (0, 4), "Vertical dipole wake potential (MV/mm)"),
]:
    zw, W = wake.wake_potential(profile, key=key)
    scale = 1e-6 if key == (0, 0) else 1e-9
    ax.plot(zw * 1e6, W * scale, "C0")
    ax.set_xlabel(r"$z$ (µm)   [head at right]")
    ax.set_ylabel(label, color="C0")
    ax2 = ax.twinx()
    ax2.fill_between(z * 1e6, current, alpha=0.2, color="C1")
    ax2.set_ylabel("Current (A)", color="C1")
plt.tight_layout()

## Applying to a ParticleGroup

`ParticleGroup.apply_wakefield` accepts a `TaylorWakefield` and applies
longitudinal *and* transverse kicks (`pz`, `px`, `py`). No `length` argument is
given, since the structure length is part of the wake table.

In [ ]:
n = 20_000
P = pg_from_random_normal(
    n,
    mean=[0, 0, 0, 0, 0, 6e9],  # 6 GeV/c
    cov=np.diag(np.array([10e-6, 1, 10e-6, 1, 10e-6, 1e-6 * 6e9]) ** 2),
)
P.charge = 250e-12  # 250 pC

P2 = P.apply_wakefield(wake)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(P.z * 1e6, (P2.pz - P.pz) * 1e-6, s=1, alpha=0.2)
axes[0].set_xlabel(r"$z$ (µm)   [head at right]")
axes[0].set_ylabel(r"$\Delta p_z$ (MeV/c)")
axes[0].set_title("Energy loss / chirp")

axes[1].scatter(P.z * 1e6, (P2.py - P.py) * 1e-6, s=1, alpha=0.2)
axes[1].set_xlabel(r"$z$ (µm)")
axes[1].set_ylabel(r"$\Delta p_y$ (MeV/c)")
axes[1].set_title("Transverse kick (toward the near plate)")
plt.tight_layout()

The head of the bunch (larger $z$) is unaffected (causality), the core and tail
lose energy, and the whole bunch is kicked toward the nearer plate, with the kick
growing toward the tail — the expected dechirper behavior.

The relative energy change and induced chirp:

In [ ]:
dp = P2.pz - P.pz
print(f"Mean energy loss: {-np.average(dp, weights=P.weight)*1e-6:.2f} MeV")
print(f"Mean vertical kick: {np.average(P2.py-P.py, weights=P.weight)*1e-6:.3f} MeV/c")

## Wake table files

`TaylorWakefield` reads and writes the ocelot/Zagorodnov numeric wake table
format, so tables can be exchanged with ocelot and with I. Zagorodnov's ECHO
family of codes (e.g. the European XFEL `*_WAKE_TAYLOR.dat` tables).

In [ ]:
wake.to_file("parallel_plate_wake_table.dat")
wake2 = TaylorWakefield.from_file("parallel_plate_wake_table.dat")
wake2

In [ ]:
# The reloaded table gives identical kicks
dpx, dpy, dpz = wake.particle_kicks_3d(P.x, P.y, P.z, P.weight)
dpx2, dpy2, dpz2 = wake2.particle_kicks_3d(P.x, P.y, P.z, P.weight)
np.allclose(dpz, dpz2, rtol=1e-12), np.allclose(dpy, dpy2, rtol=1e-12)

## Mode-sum wake for a beam near a single plate

For a beam close to one plate of a wide dechirper (large gap), the finite plate
width matters. `TaylorWakefield.dechirper_off_axis` sums transverse modes of the
corrugated structure (a port of ocelot's `WakeTableDechirperOffAxis`, based on
[Zagorodnov, Bane, Stupakov (2016)](https://doi.org/10.1016/j.nima.2016.09.001)).

In [ ]:
wake_single_plate = TaylorWakefield.dechirper_off_axis(
    plate_distance=500e-6,  # beam 500 µm from the plate
    half_gap=0.01,  # plates far apart: single-plate regime
    width=0.02,
    corrugation_gap=250e-6,
    corrugation_period=500e-6,
    length=1.0,
    sigma=10e-6,
    orientation="horizontal",
)

P3 = P.apply_wakefield(wake_single_plate)

plt.scatter(P.z * 1e6, (P3.py - P.py) * 1e-3, s=1, alpha=0.2)
plt.xlabel(r"$z$ (µm)   [head at right]")
plt.ylabel(r"$\Delta p_y$ (keV/c)")
plt.title("Kick toward a single corrugated plate")

## Cleanup

In [ ]:
import os

os.remove("parallel_plate_wake_table.dat")